# 📚 Step 3 — Recipe Matching Engine (RAG with FAISS)
### Multimodal Food Recognition & Nutrition Assistant

**Goal:** Given a food name (from CLIP in Step 2), find the most relevant recipes
from your recipes dataset using semantic search.

**What is RAG?**
Retrieval-Augmented Generation — instead of asking an LLM to invent recipes,
we retrieve REAL recipes from your dataset and pass them as context to the LLM.
This makes the chatbot grounded, accurate, and dataset-specific.

**Pipeline:**
```
Food Name ──► Sentence Embedding ──► FAISS Index Search
                                              ↓
                                   Top-K Similar Recipes
                                              ↓
                                   Context for LLM (Step 4)
```

## ⚙️ 3.1 — Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu numpy pandas
print('✅ Dependencies installed')

## 📦 3.2 — Import Libraries & Mount Drive

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# ─── CONFIGURE YOUR PATH ─────────────────────────────────────────────────
BASE       = '/content/drive/MyDrive/'                   # Same as Steps 1 & 2
OUTPUT_DIR = BASE + 'nutrition_assistant_outputs/'
# ─────────────────────────────────────────────────────────────────────────

print('✅ Libraries ready')

## 📥 3.3 — Load Cleaned Recipes from Step 1

In [ ]:
recipes_df = pd.read_csv(OUTPUT_DIR + 'recipes_clean.csv')
recipes_df = recipes_df.dropna(subset=['full_text']).reset_index(drop=True)

print(f'✅ Recipes loaded: {len(recipes_df):,} recipes')
print(f'   Columns: {list(recipes_df.columns)}')
print()
print('Sample recipe:')
print(f'  Title      : {recipes_df.iloc[0]["title"]}')
print(f'  Ingredients: {str(recipes_df.iloc[0]["ingredients_text"])[:120]}...')
print(f'  Full text  : {recipes_df.iloc[0]["full_text"][:150]}...')

## 🤖 3.4 — Load Sentence Embedding Model

In [ ]:
# all-MiniLM-L6-v2: fast, lightweight, excellent for semantic similarity
# Downloads ~90MB, cached after first run
print('Loading sentence embedding model...')

EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')

print('✅ Embedding model ready')
print(f'   Embedding dimension: {EMBED_MODEL.get_sentence_embedding_dimension()}')

## ⚡ 3.5 — Build FAISS Index (Semantic Search Engine)

In [ ]:
FAISS_INDEX_PATH   = OUTPUT_DIR + 'recipe_faiss.index'
EMBEDDINGS_PATH    = OUTPUT_DIR + 'recipe_embeddings.npy'

# Check if index already exists (skip recomputing if re-running)
if os.path.exists(FAISS_INDEX_PATH) and os.path.exists(EMBEDDINGS_PATH):
    print('📂 Found existing FAISS index. Loading...')
    index            = faiss.read_index(FAISS_INDEX_PATH)
    recipe_embeddings = np.load(EMBEDDINGS_PATH)
    print(f'✅ Index loaded: {index.ntotal:,} recipes indexed')

else:
    print(f'Building FAISS index for {len(recipes_df):,} recipes...')
    print('This may take a few minutes on first run.')

    # Encode all recipe full_text fields into embeddings
    recipe_embeddings = EMBED_MODEL.encode(
        recipes_df['full_text'].tolist(),
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # Normalise for cosine similarity search
    faiss.normalize_L2(recipe_embeddings)

    # Build FAISS flat index (exact search, best for < 500K items)
    DIM   = recipe_embeddings.shape[1]
    index = faiss.IndexFlatIP(DIM)   # Inner Product = cosine similarity after normalisation
    index.add(recipe_embeddings)

    # Save for reuse
    faiss.write_index(index, FAISS_INDEX_PATH)
    np.save(EMBEDDINGS_PATH, recipe_embeddings)

    print(f'✅ FAISS index built and saved: {index.ntotal:,} recipes indexed')

## 🔧 3.6 — Core Recipe Search Functions

In [ ]:
def search_recipes(query: str, top_k: int = 5) -> pd.DataFrame:
    """
    Search for recipes semantically similar to the query.

    Args:
        query  : food name or natural language query
                 e.g. 'chicken curry', 'low sugar dessert with banana'
        top_k  : number of recipes to return

    Returns:
        DataFrame of top_k matching recipes with similarity scores
    """
    # Encode query
    query_embed = EMBED_MODEL.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embed)

    # Search FAISS index
    scores, indices = index.search(query_embed, top_k)

    # Build results DataFrame
    results = recipes_df.iloc[indices[0]].copy()
    results['similarity_score'] = scores[0]
    results = results.reset_index(drop=True)

    return results[['title', 'ingredients_text', 'similarity_score']]


def format_recipes_for_llm(query: str, top_k: int = 3) -> str:
    """
    Returns a formatted string of top matching recipes.
    This is passed as context to the LLM chatbot in Step 4.

    Args:
        query : food name or question
        top_k : number of recipes to include as context

    Returns:
        Formatted string with recipe titles and ingredients
    """
    results = search_recipes(query, top_k=top_k)

    if results.empty:
        return 'No matching recipes found in the database.'

    context = f'TOP {top_k} MATCHING RECIPES FOR "{query.upper()}":\n'
    context += '─' * 50 + '\n'

    for i, row in results.iterrows():
        context += f'Recipe {i+1}: {row["title"]}\n'
        context += f'Ingredients: {row["ingredients_text"]}\n'
        context += f'Match Score: {row["similarity_score"]:.3f}\n\n'

    return context.strip()


def suggest_healthier_recipes(food_name: str, avoid: list = None) -> str:
    """
    Find healthier recipe alternatives for a given food.
    Optionally exclude foods to avoid (e.g. ['sugar', 'cream', 'butter']).

    Args:
        food_name : identified food name
        avoid     : list of ingredients to avoid

    Returns:
        Formatted string of healthier alternatives
    """
    query   = f'healthy low calorie {food_name} recipe'
    results = search_recipes(query, top_k=5)

    if avoid:
        # Filter out recipes containing avoided ingredients
        avoid_lower = [a.lower() for a in avoid]
        def has_avoided(ingredients):
            ing_lower = str(ingredients).lower()
            return any(a in ing_lower for a in avoid_lower)
        results = results[~results['ingredients_text'].apply(has_avoided)]

    if results.empty:
        return f'No healthier alternatives found for {food_name}.'

    context  = f'HEALTHIER ALTERNATIVES FOR "{food_name.upper()}":\n'
    context += '─' * 50 + '\n'
    for i, row in results.head(3).iterrows():
        context += f'Option {i+1}: {row["title"]}\n'
        context += f'Ingredients: {row["ingredients_text"]}\n\n'

    return context.strip()


print('✅ Recipe search functions ready')

## 🧪 3.7 — Test: Basic Recipe Search

In [ ]:
print('═' * 55)
print('TEST 1: Search for chicken recipes')
print('═' * 55)
results = search_recipes('chicken', top_k=5)
display(results)

print()
print('═' * 55)
print('TEST 2: Search for low sugar dessert')
print('═' * 55)
results = search_recipes('low sugar dessert chocolate', top_k=5)
display(results)

## 🧪 3.8 — Test: Format Recipes as LLM Context

In [ ]:
print('═' * 55)
print('TEST 3: LLM context block for "pizza"')
print('═' * 55)
context = format_recipes_for_llm('pizza', top_k=3)
print(context)

print()
print('═' * 55)
print('TEST 4: Healthier alternatives for "fried chicken"')
print('═' * 55)
alts = suggest_healthier_recipes('fried chicken', avoid=['cream', 'butter'])
print(alts)

## 🔗 3.9 — Connect to Step 2: Image → Food → Recipes

In [ ]:
def food_name_to_recipe_context(food_name: str) -> dict:
    """
    Given a food name from CLIP (Step 2), returns a full context
    dictionary ready to be passed into the LLM chatbot (Step 4).

    Returns dict with:
      - food_name       : the identified food
      - similar_recipes : formatted top-3 matching recipes
      - healthy_alts    : healthier recipe alternatives
    """
    return {
        'food_name'       : food_name,
        'similar_recipes' : format_recipes_for_llm(food_name, top_k=3),
        'healthy_alts'    : suggest_healthier_recipes(food_name)
    }


# Test the full bridge function
context = food_name_to_recipe_context('pizza')
print(f'Food identified : {context["food_name"]}')
print()
print(context['similar_recipes'])
print()
print(context['healthy_alts'])

## 💾 3.10 — Save Everything

In [ ]:
# FAISS index and embeddings already saved in 3.5
# Save a small metadata file for Step 4 to reference

meta = {
    'embed_model_name' : 'all-MiniLM-L6-v2',
    'num_recipes'      : len(recipes_df),
    'faiss_index_path' : FAISS_INDEX_PATH,
    'embeddings_path'  : EMBEDDINGS_PATH,
}

with open(OUTPUT_DIR + 'rag_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('✅ Saved files:')
print(f'   recipe_faiss.index      ← FAISS search index')
print(f'   recipe_embeddings.npy   ← raw embeddings')
print(f'   rag_meta.pkl            ← metadata for Step 4')

## ✅ Step 3 Complete!

| Capability | Status |
|---|---|
| Sentence embeddings for all recipes | ✅ |
| FAISS semantic search index | ✅ |
| `search_recipes(query)` | ✅ |
| `format_recipes_for_llm(query)` | ✅ |
| `suggest_healthier_recipes(food)` | ✅ |
| `food_name_to_recipe_context(food)` | ✅ |

**Key output for Step 4:**
- `format_recipes_for_llm(food_name)` → ready-to-inject LLM context block
- `suggest_healthier_recipes(food_name)` → healthier alternatives block

---
**➡️ Next: Step 4 — Nutrition Q&A Chatbot (LLM layer) + Gradio Demo App**